# Week 4 · Try It 1: Cluster a Real LiDAR Scan

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/weisongwen/AAE4011-S22627/blob/main/Lecture%20Notes/Week%204/lidar_clustering.ipynb)

AAE4011 Artificial Intelligence in Unmanned Autonomous Systems · Semester 2, 2026/27

One scan of a 32-beam Velodyne LiDAR on a car roof, taken from last year's Week 3–4 ROS bag (`/velodyne_points`). The file `lidar_scan.npz` (0.6 MB) holds about 55 000 points: `xyz` in metres (x forward, y left, z up, sensor at the origin), `intensity` and `ring`.

The pipeline is the one on the slide *UAS Case 1: Segmenting a Real LiDAR Scan into Obstacles*: crop, remove the ground, then DBSCAN on the (x, y) positions.

**Questions** (the same as on the slide)
- (a) How many clusters are there at ε = 0.5 m, minPts = 10?
- (b) What happens at ε = 0.2 m and at 1.0 m?
- (c) Why does K-means with K = 31 cut the long wall?
- (d) Which clusters are cars? How can you tell?

In scikit-learn, minPts is called `min_samples`; as in the lecture, a point counts itself.

In [ ]:
import os, urllib.request
URL = "https://raw.githubusercontent.com/weisongwen/AAE4011-S22627/main/Lecture%20Notes/Week%204/"
if not os.path.exists("lidar_scan.npz"):
    urllib.request.urlretrieve(URL + "lidar_scan.npz", "lidar_scan.npz")
print("ready:", os.path.getsize("lidar_scan.npz") // 1024, "KB")

## Part 1 · Crop and remove the ground

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

xyz = np.load("lidar_scan.npz")["xyz"].astype(float)
print("points in the file:", len(xyz))

# 1  crop: a 40 m x 40 m square around the sensor, without the points on our own car
R = 20.0
p = xyz[(np.abs(xyz[:, 0]) < R) & (np.abs(xyz[:, 1]) < R)]
ego = (np.abs(p[:, 0]) < 1.3) & (np.abs(p[:, 1]) < 3.0)
p = p[~ego]

# 2  ground: the most common height within 12 m of the sensor; keep 0.3 to 2.5 m above it
near = xyz[np.linalg.norm(xyz[:, :2], axis=1) < 12]
h, edges = np.histogram(near[:, 2], bins=np.arange(-4, 2, 0.05))
g = edges[np.argmax(h)] + 0.025
obj = p[(p[:, 2] > g + 0.3) & (p[:, 2] < g + 2.5)]
print(f"ground at z = {g:.2f} m (the sensor is {-g:.2f} m above it)")
print(f"{len(p)} points within {R:g} m, {len(obj)} object points after ground removal")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
ax[0].scatter(p[:, 0], p[:, 1], s=0.2, c=np.clip(p[:, 2] - g, -0.2, 2.5), cmap="viridis")
ax[0].set_title("raw scan, coloured by height")
ax[1].scatter(p[:, 0], p[:, 1], s=0.1, c="0.85")
ax[1].scatter(obj[:, 0], obj[:, 1], s=0.3, c="navy")
ax[1].set_title("object points (ground removed)")
for a in ax:
    a.plot(0, 0, "r^"); a.set_aspect("equal"); a.set_xlabel("x (m)")
ax[0].set_ylabel("y (m)")
plt.show()

## Part 2 · DBSCAN

Change `eps` and `min_samples` and run the cell again.

In [ ]:
from sklearn.cluster import DBSCAN

eps, min_samples = 0.5, 10          # change these and run again
lab = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(obj[:, :2])   # cluster on (x, y) only
print(f"eps = {eps} m, min_samples = {min_samples}: {lab.max() + 1} clusters, "
      f"{np.sum(lab < 0)} noise points")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(p[:, 0], p[:, 1], s=0.1, c="0.88")
ax.scatter(obj[lab < 0, 0], obj[lab < 0, 1], s=1, c="0.4")
ax.scatter(obj[lab >= 0, 0], obj[lab >= 0, 1], s=0.6, c=lab[lab >= 0] % 10, cmap="tab10")
ax.plot(0, 0, "r^"); ax.set_aspect("equal"); ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
ax.set_title(f"DBSCAN: {lab.max() + 1} clusters (grey = noise)")
plt.show()

In [ ]:
for e in (0.2, 0.5, 1.0):
    l = DBSCAN(eps=e, min_samples=10).fit_predict(obj[:, :2])
    print(f"eps = {e:3.1f} m: {l.max() + 1:3d} clusters, {np.sum(l < 0):4d} noise points")

**Questions (a), (b).** Why does a small ε break objects into pieces and leave many noise points? Why does a large ε merge neighbouring objects?

## Part 3 · Choose ε with the k-distance plot

In [ ]:
from sklearn.neighbors import NearestNeighbors

k = 10                                   # = min_samples; each point is its own first neighbour
dist, _ = NearestNeighbors(n_neighbors=k).fit(obj[:, :2]).kneighbors(obj[:, :2])
dk = np.sort(dist[:, -1])                # distance to the 9th other point, sorted
plt.plot(100 * np.arange(1, len(dk) + 1) / len(dk), dk)
plt.axhline(0.5, color="r", ls="--", label="eps = 0.5 m")
plt.xlim(80, 100); plt.ylim(0, 1.5)
plt.xlabel("object points, sorted (%)"); plt.ylabel("distance to 9th neighbour (m)")
plt.legend(); plt.show()
print(f"{100 * np.mean(dk < 0.5):.1f} % of the points have their 9th neighbour within 0.5 m")

## Part 4 · K-means with the 'right' K, and which clusters are cars

We give K-means the number of objects DBSCAN found and compare the two. The table lists the 12 objects nearest to the sensor: footprint, height of the top above the ground, and into how many K-means clusters each object is cut.

In [ ]:
from sklearn.cluster import KMeans

ref = DBSCAN(eps=0.5, min_samples=10).fit_predict(obj[:, :2])     # the reference objects
K = ref.max() + 1
km = KMeans(n_clusters=K, n_init=10, random_state=0).fit_predict(obj[:, :2])

rows = []
for k in range(K):
    c = obj[ref == k]
    cx, cy = c[:, :2].mean(axis=0)
    rows.append((np.hypot(cx, cy), k, np.ptp(c[:, 0]), np.ptp(c[:, 1]), c[:, 2].max() - g, len(c),
                 len(np.unique(km[ref == k]))))
print(f"K = {K}")
print("object  distance  footprint (m)  top (m)  points  K-means pieces")
for r, k, dx, dy, top, n, pieces in sorted(rows)[:12]:
    print(f"  {k:3d}    {r:5.1f} m    {dx:4.1f} x {dy:4.1f}    {top:4.1f}    {n:5d}        {pieces}")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
for ax, l, title in ((axs[0], km, f"K-means, K = {K}"), (axs[1], ref, "DBSCAN, eps = 0.5 m")):
    ax.scatter(p[:, 0], p[:, 1], s=0.1, c="0.88")
    m = l >= 0
    ax.scatter(obj[~m, 0], obj[~m, 1], s=1, c="0.4")
    ax.scatter(obj[m, 0], obj[m, 1], s=0.8, c=l[m] % 10, cmap="tab10")
    ax.plot(0, 0, "r^"); ax.set_aspect("equal"); ax.set_xlim(-11, 7); ax.set_ylim(-11, 3)
    ax.set_title(title); ax.set_xlabel("x (m)")
plt.show()

**Questions (c), (d).** Find the 20.8 m wall in the table: into how many pieces does K-means cut it, and why? Which rows are cars? (A car is about 2 m × 4.5 m and 1.5 m tall.)

---
## Check your answers (try first!)

- (a) 31 clusters, 173 noise points.
- (b) ε = 0.2 m: 72 clusters, 884 noise points (objects fall apart); ε = 1.0 m: 19 clusters, 57 noise points (neighbouring objects merge). The k-distance plot has its knee near 0.5 m: 98.4 % of the points have their 9th neighbour closer than that.
- (c) The wall is cut into 6 pieces and the nearest car into 3. K-means puts every point with its nearest centre, so it prefers round clusters of similar size; a long thin wall must be shared by several centres.
- (d) Three clusters: 1.9 × 4.5 m, 1.7 × 3.5 m and 2.3 × 3.4 m, each with its top about 1.5 m above the ground. A wall is long and thin; a pole is small; a car is a 2 × 4 m box about 1.5 m tall.

With another scikit-learn version the K-means pieces can differ slightly; the DBSCAN numbers do not.